# 1. 모듈과 패키지?

# 2. import 문법

In [1]:
import math

print(math.pi)
print(math.sqrt(16))
print(math.floor(3.7))

3.141592653589793
4.0
3


# 3. import ... as

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

arr = np.array([1, 2, 3])
df = pd.DataFrame({'a': [1, 2]})

print(arr)
print(df)

[1 2 3]
   a
0  1
1  2


# 4. from ... import ...

In [4]:
from math import pi, sqrt
from datetime import datetime

print(pi)
print(sqrt(25))
print(datetime.now())

3.141592653589793
5.0
2026-08-27 09:33:05.942047


# 5. from ... import *

In [ ]:
from math import *
print(pi, sqrt(9), floor(4.8))

# 6. 모듈 탐색 경로와 실행 방식

In [ ]:
import sys
print(sys.path)

# 7. 사용자 정의 모듈 만들기

``` python
%%writefile sensor_utils.py

def is_abnormal(value, limit):
    """기준값 초과 여부 판단"""
    return value > limit

PI = 3.14159 # 모듈에는 변수도 담을 수 있음
```

``` python
# 파일: main.py (같은 폴더)
from sensor_utils import is_abnormal, PI

print(is_abnormal(32, 30)) # True
print(PI)
```

# 8. \_\_name\_\_ = "\_\_main\_\_"

``` python
# 파일: sensor_utils.py
def is_abnormal(value, limit):
    return value > limit

if __name__ == '__main__':
    # 이 파일을 직접 실행할 때만 아래 코드가 동작
    print(is_abnormal(35, 30)) # 테스트/디버깅용 코드
```

# 9. 패키지 구조 만들기

``` python
# 폴더 구조
# project/
# ├─ sensors/
# │ ├─ __init__.py
# │ ├─ converter.py
# │ └─ validator.py
# └─ main.py
# main.py

from sensors.converter import c_to_f
from sensors import validator

```

In [10]:
# 폴더 생성
from pathlib import Path

project_dir = Path.cwd() / 'project'
sensors_dir = project_dir / 'sensors'

sensors_dir.mkdir (
    parents=True,
    exist_ok=True
)

print("프로젝트: ", project_dir.resolve())
print("패키지: ", sensors_dir.resolve())

프로젝트:  C:\Users\User\Documents\jupyter\project
패키지:  C:\Users\User\Documents\jupyter\project\sensors


In [24]:
# 파일 생성
converter_code = '''def c_to_f(celsius) :
    """섭씨를 화씨로 변환합니다."""
    return float(celsius) * 9 / 5 + 32
    '''
(sensors_dir/'converter.py').write_text(  #converter.py를 저장합니다.
    converter_code, encoding='utf-8')

validator_code = '''def is_valid_temperature (value):
    """숫자로 변환 가능한 값인지 검사합니다."""
    try:
        float(value)
        return True
    except (TypeError, ValueError):
        return False
'''

(sensors_dir/'validator.py').write_text(#validator.py를 저장합니다.
validator_code, encoding='utf-8')

173

In [14]:
# init
init_code = ''' #c_to_f 함수를 패키지 수준으로 공개합니다.
from .converter import c_to_f

# validator 모듈을 패키지 수준으로 공개합니다.
from . import validator
'''

(sensors_dir/'_init_.py').write_text(#_init_py 파일을 생성합니다.
    init_code,
    encoding='utf-8'
)

print('_init_ py 생성 완료')
# 주의: 파일명은 init.py입니다.
# init 앞뒤에 밑줄이 각각 두 개씩 필요합니다.

_init_ py 생성 완료


In [18]:
# 검색 경로 등록, import 캐시 초기화
import sys # 모듈 검색 경로를 관리합니다.
import importlib # import 캐시를 갱신합니다.

project_path = str(project_dir.resolve())

if project_path in sys.path:
    sys.path.remove(project_path)
sys.path.insert(0, project_path)

# project의 전체 경로를 문자열로 만듭니다.
# 기존 검색 경로에 있으면 # 중복 항목을 먼저 제거합니다.
#검색 경로의 첫 번째 위치에 등록합니다.

cached = [name for name in sys.modules   #메모리에 저장된 sensors 모듈을 찾습니다.
          if name == 'sensors' or name.startswith('sensors.')]

for name in cached: # 찾은 모듈을 하나씩
    del sys.modules[name]
# 기존 import 캐시에서 제거합니다.

importlib.invalidate_caches()
print('첫 검색 경로:', sys.path[0])
#새 파일을 다시 검색하도록 갱신합니다.

첫 검색 경로: C:\Users\User\Documents\jupyter\project


In [25]:
# 최종 프로그램 실행과 결과
from sensors.converter import c_to_f 
from sensors import validator 
import sensors
#섭씨,화씨 함수를 가져옵니다.
#validator 모듈을 가져옵니다.
# 실제 패키지 위치 확인용으로 가져옵니다.

print('패키지 위치:', sensors.__file__) # project/sensors인지 반드시 확인합니다.
temperature = 20
# 검사하고 변환할 온도입니다.

if validator.is_valid_temperature(temperature):    #숫자로 변환 가능하면
    fahrenheit = c_to_f(temperature)
    # 섭씨를 화씨로 변환합니다.
    print(f'섭씨 {temperature}C = 화씨 {fahrenheit}F')
else:
    print("온도에 숫자를 입력하세요.")
# 숫자가 아니면 입력 오류를 안내합니다.
# 정상 결과: 섭씨 20°C = 화씨 68.0°F

패키지 위치: None
섭씨 20C = 화씨 68.0F


# 10. __init__.py

``` python
# 파일: sensors/__init__.py
from .converter import c_to_f # 패키지 레벨에서 바로 노출

# main.py에서는 아래처럼 짧게 사용 가능
from sensors import c_to_f
print(c_to_f(20))

```

# 11. 절대/상대 import

# 12. 외부 패키지 설치와 버전 확인

In [27]:
# 터미널 또는 Notebook 셀에서 실행
# conda install numpy pandas -y
# 또는: pip install numpy pandas
import numpy as np
import pandas as pd
print('NumPy:', np.__version__)
print('Pandas:', pd.__version__)


NumPy: 2.4.6
Pandas: 3.0.3


# 13. dir/help

In [28]:
import math
print(dir(math)) # math 모듈이 제공하는 이름 전체 목록
help(math.sqrt) # sqrt 함수의 사용법(docstring) 출력


['__doc__', '__loader__', '__name__', '__package__', '__spec__', 'acos', 'acosh', 'asin', 'asinh', 'atan', 'atan2', 'atanh', 'cbrt', 'ceil', 'comb', 'copysign', 'cos', 'cosh', 'degrees', 'dist', 'e', 'erf', 'erfc', 'exp', 'exp2', 'expm1', 'fabs', 'factorial', 'floor', 'fma', 'fmod', 'frexp', 'fsum', 'gamma', 'gcd', 'hypot', 'inf', 'isclose', 'isfinite', 'isinf', 'isnan', 'isqrt', 'lcm', 'ldexp', 'lgamma', 'log', 'log10', 'log1p', 'log2', 'modf', 'nan', 'nextafter', 'perm', 'pi', 'pow', 'prod', 'radians', 'remainder', 'sin', 'sinh', 'sqrt', 'sumprod', 'tan', 'tanh', 'tau', 'trunc', 'ulp']
Help on built-in function sqrt in module math:

sqrt(x, /)
    Return the square root of x.



# 14. 프로젝트 의존성 관리

In [31]:
# requirements.txt 파일 내용 예시
# numpy==1.26.4
# pandas==2.2.2

# 설치 명령
# pip install -r requirements.txt

# 현재 환경의 패키지 목록 저장
# pip freeze > requirements.txt

# 예제

In [32]:
# 예제 1번
import math

radius = 5

print("원 넓이: ", math.pi * radius ** 2)
print("원 둘레: ", 2 * math.pi * radius)

원 넓이:  78.53981633974483
원 둘레:  31.41592653589793


In [34]:
# 예제 2번
## 파일 생성
from pathlib import Path

cur_dir = Path.cwd()

sensor_utils_code = '''def is_abnormal(value, limit) :
    """측정값이 기준치를 초과하면 True 반환"""
    return value > limit
'''

(cur_dir/'sensor_utils.py').write_text( 
    sensor_utils_code, encoding='utf-8')

90

In [36]:
# 예제 2번
from sensor_utils import is_abnormal

temp = 34.5
limit = 30

if is_abnormal(temp, limit):
    print("경고: 기준치 초과")
else:
    print("정상 범위임")

경고: 기준치 초과


In [37]:
# 예제 3번
## 파일 생성
from pathlib import Path

cur_dir = Path.cwd()

sensor_utils_code = '''def is_abnormal(value, limit) :
    """측정값이 기준치를 초과하면 True 반환"""
    return value > limit

if __name__ == '___main___':
    print('테스트:', is_abnormal(35, 30)) # True
    print('테스트:', is_abnormal(20, 30)) # False
'''

(cur_dir/'sensor_utils.py').write_text( 
    sensor_utils_code, encoding='utf-8')

213